# L3b: Stacks and Queues

An array lets you read or write any element at any time. A stack and a queue deliberately give up that freedom: items enter and leave only at the ends, and which end you may touch decides the order the structure hands work back to you. Today we build both structures on top of ordinary Julia arrays, wrap them in interfaces that enforce the rules, and meet the places they are already running your programs. Before we close, we meet a third structure, the linked list, which stores a sequence in references rather than positions and is the seed of the trees ahead.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Match the structure to the problem:__ Explain what last-in-first-out and first-in-first-out mean, and pick a stack or a queue from the order a problem must honor. Contrast an array, which stores order in positions, with a linked list, which stores order in references from one node to the next, and say what each layout makes cheap.
> * __Operate stacks and queues through a public interface:__ Add and remove items only through the functions a type provides, and explain how keeping the backing array private by convention protects the promised order. This is the same public-interface, private-implementation split that a well-designed function contract gave us in Week 2.
> * __Recognize stacks and queues inside running programs:__ Identify the call stack that runs every Julia function as a stack of frames, and the frontier of next week's breadth-first graph search as a queue. Knowing which structure an algorithm depends on tells you the order it will do its work before you run it.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

This lecture needs nothing beyond the course environment: the [Test standard library](https://docs.julialang.org/en/v1/stdlib/Test/) supplies the checks we run along the way, the `L3bStacksQueues` module in [`src/Compute.jl`](src/Compute.jl) supplies the `Stack` and `Queue` types and [the `isbalanced(...)` function](src/Compute.jl) we examine, and the `MutableLinkedList` type in the linked-list section comes from [the DataStructures.jl package](https://github.com/JuliaCollections/DataStructures.jl), which the course environment already carries.
___

## Stacks: last in, first out

Some work is naturally served most-recent-first. The undo feature of an editor reverses the latest edit, not the oldest one, and the back button of a browser returns to the page you just left. The structure behind every one of these is a stack.

> __Stack discipline (LIFO):__
>
> A stack accepts new items and releases items at the same end, called the top. The last item pushed is the first item popped, so a stack replays history in reverse. This order is called __last-in-first-out__, or __LIFO__. Think of a stack of plates: you can only add or remove the top plate, and the last plate you put on is the first one you take off.

The schematic traces the discipline on the values we will reuse all lecture: `push!(s, 16)` places 16 on top of the stack holding 8, 4, 2, and the very next `pop!(s)` removes and returns that same 16, the newest item.

<div>
    <center>
        <img src="figs/Fig-Stack.svg" width="560" alt="A stack holding 8, 4, 2 shown before and after push!(s, 16) places 16 on top, and after pop!(s) removes and returns that same 16"/>
    </center>
</div>

A plain Julia vector already supports the two motions: [the `push!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.push!) appends an item at the back, and [the `pop!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.pop!) removes and returns the item at the back. Treat the back of the vector as the top of the stack, and undo falls out by itself. Let's record three edits and then undo two of them:

In [ ]:
let
    edit_history = Vector{String}() # the array behind an undo feature
    push!(edit_history, "typed the heading")
    push!(edit_history, "added the figure")
    push!(edit_history, "fixed the caption")
    (first_undo = pop!(edit_history), second_undo = pop!(edit_history), still_applied = edit_history)
end

The two undo steps came back in reverse order of entry: the caption fix first, then the figure, with the heading still applied. That is LIFO doing exactly what an undo feature needs.

There is a catch, though. Nothing about a plain vector enforces the discipline: any caller can reach around the stack story, remove an element from the front, and the LIFO guarantee silently dies. The `L3bStacksQueues` module in [`src/Compute.jl`](src/Compute.jl) closes that hole with a `Stack` type: a [composite type](https://docs.julialang.org/en/v1/manual/types/#Composite-Types) holding the vector in a private `items` field, with public functions for pushing, popping, and peeking as the only supported access path. Julia does not lock fields away, so this is a convention rather than a wall, but it is the convention every Julia library leans on: code that stays on the supported path cannot break the order.

Let's point the `Stack` type at a job from Week 2, walking the characters of a string, and chemistry hands us a good string to walk: the molecular formula of glucose. The `formula_stack::Stack{Char}` variable holds each character of `C6H12O6` in arrival order:

In [ ]:
formula_stack = let
    formula_stack = Stack{Char}()
    for character in "C6H12O6"
        push!(formula_stack, character)
    end
    formula_stack
end

Popping until the stack is empty must now return the characters in reverse arrival order. The `reversed_formula::String` variable collects them:

In [ ]:
reversed_formula = let
    characters = Vector{Char}()
    while !isempty(formula_stack)
        push!(characters, pop!(formula_stack))
    end
    String(characters)
end

The formula comes back as `6O21H6C`: the final `6` was pushed last, so it is popped first. A stack is the wrong way to read a molecular formula, and that wrongness is the point. Reversal is the definition of LIFO made visible, and it is the reason a stack is the right home for anything that must unwind in the opposite order it was built. Reading the formula back intact takes the other discipline.
___

## Queues: first in, first out
Other work must be served in arrival order. A shared printer takes jobs in the order they were submitted, and a simulation processes events in the order they occur. Serving either of these most-recent-first would be wrong in an obvious way: latecomers would jump the line.

> __Queue discipline (FIFO):__
>
> A queue accepts new items at the back and releases items from the front. The first item in is the first item out, so a queue preserves arrival order. This order is called __first-in-first-out__, or __FIFO__.

The schematic runs the same values through this discipline: pushing 16 adds it at the back of the line behind 8, while the next removal serves the 2 at the front, exactly as a checkout line would.

<div>
    <center>
        <img src="figs/Fig-Queue.svg" width="500" alt="A queue holding 2, 4, 8 shown before and after push!(q, 16) joins the back of the line, and after popfirst!(q) serves and returns the 2 from the front"/>
    </center>
</div>

On a plain vector the front motion is [the `popfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.popfirst!), which removes and returns the first element. The `Queue` type in [`src/Compute.jl`](src/Compute.jl) wraps that discipline the same way `Stack` wrapped LIFO: internal vector, public functions, no other supported access path. It also settles the glucose question from the stack section. The `formula_reading::String` variable pushes the same formula through a `Queue{Char}` and drains it from the front:

In [ ]:
formula_reading = let
    character_queue = Queue{Char}()
    for character in "C6H12O6"
        push!(character_queue, character)
    end
    characters = Vector{Char}()
    while !isempty(character_queue)
        push!(characters, popfirst!(character_queue))
    end
    String(characters)
end

The queue reads the formula back exactly as written, `C6H12O6`, where the stack returned `6O21H6C`. Same characters, same backing array, opposite readings; the only difference between the two runs is which end each discipline is allowed to touch.

> __What the front end costs:__
>
> Removing the front element of a vector shifts every remaining element left by one slot, so dequeuing from a long vector-backed queue does work proportional to its length. That is perfectly fine at classroom scale. Production code reaches for a purpose-built structure such as the `Deque` type from [the DataStructures.jl package](https://github.com/JuliaCollections/DataStructures.jl), which serves both ends cheaply. We will meet this build-versus-buy trade again on Thursday, when our hand-built sorts race Julia's library sort.

___

## Application: checking balanced delimiters
Every Julia expression you have typed this semester obeyed a rule you never checked by hand: parentheses, brackets, and braces must close in last-opened-first-closed order. Read that phrase again, because last-opened-first-closed is stack discipline, and a stack is exactly how a parser verifies delimiters.

> __The algorithm:__
>
> Walk the text one character at a time, keeping a stack of the delimiters currently open. An opener is pushed. A closer must match the most recently opened delimiter, so pop the stack and compare. A mismatch, a closer with nothing open, or anything left open at the end of the text each mean the text is unbalanced.

The `isbalanced(...)` function in [`src/Compute.jl`](src/Compute.jl) implements this walk, and it shows the public-interface, private-implementation split in its smallest form: `isbalanced(...)` is the public face, while the `_OPENER_FOR_CLOSER` table and [the `_isopener(...)` helper function](src/Compute.jl) are private details, marked by the leading-underscore naming convention Julia programmers use for internals a caller should not rely on. One honest caveat: the checker reads raw characters, so a lone delimiter inside a quoted string, printed to the screen by [the `println(...)` function](https://docs.julialang.org/en/v1/base/io-network/#Base.println) say, counts like any other; a real parser strips string literals and comments before a check like this runs. Let's exercise the public face with [the `@testset` macro](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@testset), including the failure cases that prove the checker can say no:

In [ ]:
@testset "isbalanced enforces stack discipline" begin
    @test isbalanced("f(x[2]) + {a: (b)}") # closes in last-opened-first-closed order
    @test isbalanced("no delimiters at all") # nothing to check is balanced
    @test !isbalanced("f(x[2)]") # closes the paren while the bracket is still open
    @test !isbalanced(")(") # closes a paren that was never opened
    @test !isbalanced("open( forever") # leaves a paren open at the end
end;

All five verdicts agree with a reading by eye, and the two ingredients doing the work are the ones this lecture is about: a stack to remember what is open, and an interface that keeps the caller out of the details.
___

## Coupling the disciplines: a replay buffer
The two disciplines earn their keep when they work together. Picture a survey robot exploring a planetary surface: mission control transmits movement commands, which the robot must execute in the order sent, and a `back` command must undo the most recent move. Execution order is a queue's job; undo is a stack's. One stream of commands, two disciplines, each covering what the other cannot.

> __The design:__
>
> Commands wait in a `Queue{Char}` and are executed first-in-first-out, exactly as transmitted. Every executed move is also pushed onto a `Stack{Char}` of history. A `back` command pops that history and executes the inverse of whatever comes off, so undo always applies to the most recent move, no matter when the command sequence was written.

The `replay_trace::NamedTuple` variable runs a small mission: north, north, east, east, west, then two `back` commands and a pause. Watch what each `back` undoes:

In [ ]:
replay_trace = let
    command_tape = Queue{Char}()
    for command in "nneewbbp" # north, north, east, east, west, back, back, pause
        push!(command_tape, command)
    end

    displacement = Dict('n' => (0, 1), 's' => (0, -1), 'e' => (1, 0), 'w' => (-1, 0), 'p' => (0, 0))
    inverse_move = Dict('n' => 's', 's' => 'n', 'e' => 'w', 'w' => 'e')

    undo_history = Stack{Char}()
    x, y = 0, 0
    mission_log = Vector{String}()
    while !isempty(command_tape)
        command = popfirst!(command_tape) # commands execute in transmission order
        action = command
        if command == 'b'
            action = isempty(undo_history) ? 'p' : inverse_move[pop!(undo_history)]
        elseif haskey(inverse_move, command)
            push!(undo_history, command) # only real moves are undoable
        end
        (dx, dy) = displacement[action]
        x += dx
        y += dy
        push!(mission_log, "command $(command) -> action $(action), position ($(x), $(y))")
    end
    (final_position = (x, y), mission_log = mission_log)
end

The log shows the first `back` undoing the westward step and the second undoing an eastward one: most recent first, which is the stack talking. The commands themselves executed in exactly the transmitted order, which is the queue talking. This coupling, a queue for what to do and a stack for what was done, is the shape of nearly every undo system you have used.
___

## The call stack
You have been using a stack all semester without seeing it. Every time a Julia function is called, the runtime pushes a __stack frame__ holding the function's local variables and the point to return to; when the function returns, its frame is popped. Calls therefore unwind in reverse order, and the most recently entered function is always the first to finish. That structure is the __call stack__, and it is a stack in exactly the sense of this lecture.

To watch the discipline at work, let's nest three functions and print a line on the way into and out of each one:

In [ ]:
let
    inner() = println("        inner  entered last, finished first")
    function middle()
        println("    middle entered second")
        inner()
        println("    middle finished second")
    end
    function outer()
        println("outer  entered first")
        middle()
        println("outer  finished last")
    end
    outer()
end

The entry lines print in call order and the finish lines print in reverse, which is the push-pop pattern of the undo demo wearing function clothes. The frame budget is finite: a chain of calls that never returns keeps pushing frames until the runtime gives up with a [`StackOverflowError` exception](https://docs.julialang.org/en/v1/base/base/#Core.StackOverflowError), an error named for this exact stack.
___

## Linked lists: order without an array

Our `Stack` and `Queue` types kept their items in one contiguous array, and the queue's cost note showed the bill that layout can present: serving the front of a vector shifts every element behind it. A __linked list__ takes the opposite bet. Each item lives in its own small __node__, and a node holds exactly two things: a value, and a reference to the node that follows it. The list itself remembers only the __head__, the first node; the last node references nothing, which is how a traversal knows to stop.

> __Position versus reference:__
>
> An array stores order in positions: the fifth element sits five slots from the start, so jumping straight to it is instant, but making room at the front means shifting everything. A linked list stores order in references: reaching the fifth element means following four links, but splicing a node in or out at a place you already hold is a couple of relinks, with nothing shifted. Neither layout wins outright; they price the same operations differently.

<div>
    <center>
        <img src="figs/Fig-LinkedList.svg" width="700" alt="A linked list holding 2, 4, 8: the head references the first node, each node holds a value and a reference to the node that follows it, the last node references nothing, and a second row shows 16 spliced in after 4 with two relinks while nothing else moves"/>
    </center>
</div>

The schematic's splice is the operation arrays are worst at, done for the price of two reference updates, and it completes the story of the earlier figures: the stack pushed 16 on top, the queue added it at the back, and the linked list can put it anywhere. We do not need to build the structure by hand: the `MutableLinkedList` type from [the DataStructures.jl package](https://github.com/JuliaCollections/DataStructures.jl), the same package whose `Deque` the queue section mentioned, is a ready-made linked list. Let's load the glucose formula into one. The `formula_list::MutableLinkedList{Char}` variable holds one node per character, each linked after the last:

In [ ]:
formula_list = let
    formula_list = MutableLinkedList{Char}()
    for character in "C6H12O6"
        push!(formula_list, character) # each new node is linked after the current last node
    end
    formula_list
end

Iterating the list follows the references from the head, so the characters come back in arrival order, no array required. The front is also now a cheap end. The `front_edit::NamedTuple` variable pushes a stray character on with [the `pushfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.pushfirst!) and removes it with [the `popfirst!(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.popfirst!); each operation is a constant amount of pointer work, whatever the length of the list, and then we read the whole list back:

In [ ]:
front_edit = let
    pushfirst!(formula_list, '?') # add at the front: constant pointer work, nothing shifts
    removed = popfirst!(formula_list) # remove it again; the existing nodes never move
    (removed = removed, reading = String(collect(formula_list)))
end

The reading is `C6H12O6`, intact and in order, and neither front operation moved another node. This idea is where today's material grows next. A linked-list node references exactly one successor; let nodes reference several, grown from a single root with no cycles, and the chain becomes a __tree__. Tomorrow's lecture draws a recursive computation as a tree of calls, and the active path through that tree lives on the very call stack we just watched. Next week, graph search walks structures built from this node-and-reference idea, holding its frontier of unexplored vertices in a container of your choosing: make it a queue and the search spreads outward level by level, make it a stack and it dives deep before backing up. Same algorithm, different discipline, different traversal.
___

## Summary
A stack and a queue are the same array wearing two different rules about which end you may touch, and the rule you pick decides the order work comes back.

> __Key Takeaways:__
>
> * **The rule sets the order, the storage sets the cost:** Pushing and popping at one end gives last-in-first-out, entering at the back and leaving from the front gives first-in-first-out, and today both were rules applied to an ordinary Julia vector. The linked list changed the storage instead of the rule, keeping order in references from one node to the next, so a splice is two relinks but reaching an element means following the links.
> * **An interface protects the order:** Our stack and queue types route every supported operation through functions that respect the promised order and keep the backing vector internal by convention. Code that stays on the supported path cannot break the order, which is the public-interface, private-implementation contract from Week 2 applied to a whole type.
> * **Stacks and queues are already running your programs:** The call stack pushes a frame at every function call and pops it at every return, a parser's delimiter check is a stack at work, and the replay buffer coupled a queue of pending commands to a stack of executed history. Knowing which structure an algorithm depends on tells you the order it will do its work before you run it.

Pick the rule from the order a problem must honor, and pick the storage from the operations it will pay for most often.
___